# First experiment: official YOLOX + ByteTrack in Colab

Goal: run the authors' detector and tracker on a short sample and download a video with person IDs. This is a smoke test, not a benchmark reproduction or accuracy measurement. No training is required.

On your Mac, upload this notebook at https://colab.research.google.com/ using **File > Upload notebook**. Choose **Runtime > Change runtime type > a GPU**, then run the cells in order. A GPU is subject to Colab availability.

Sources: [paper](https://arxiv.org/abs/2110.06864), [official code and checkpoint links](https://github.com/FoundationVision/ByteTrack), [Colab help](https://research.google.com/colaboratory/faq.html).

This learning notebook is AI-assisted. Follow your professor's restrictions on assessed submissions. It has been checked structurally on a Mac, but has not been executed in a Colab GPU session. It uses official source with small, recorded compatibility edits for deprecated NumPy types and headless video operation. It does not use Ultralytics or YOLO11.


## 1. Check the GPU
If this fails, select a GPU runtime and rerun. Keep Colab's matching PyTorch and torchvision packages.


In [ ]:
import sys, subprocess, json, hashlib, re, os, time, shutil
from pathlib import Path
import torch, torchvision
assert torch.cuda.is_available(), 'Choose a GPU under Runtime > Change runtime type.'
print('Python:', sys.version)
print('PyTorch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))


## 2. Install demo dependencies
The old repository requirements include dated optional ONNX packages. This installs the dependencies for the PyTorch video demo instead. It does not change the tracking algorithm or replace PyTorch. If Colab requests a restart, restart and rerun from Step 1.


In [ ]:
packages = ['loguru', 'scikit-image', 'tqdm', 'Pillow', 'thop', 'ninja', 'tabulate',
            'tensorboard', 'lap', 'motmetrics', 'filterpy', 'h5py', 'pycocotools',
            'cython_bbox', 'gdown']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)


## 3. Get the official source and record compatibility edits
The source revision is fixed. Running from the source directory avoids building the optional accelerated COCO evaluator, which this demo does not use. Deprecated `np.float` aliases become `float`, the GUI key wait is disabled, and checkpoint loading explicitly uses PyTorch's restricted weights loader. All edits are saved in a patch file.


In [ ]:
REVISION = 'd1bf0191adff59bc8fcfeaa0b33d3d1642552a99'
REPO = Path('/content/ByteTrack_paper_starter')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/FoundationVision/ByteTrack.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', REVISION], check=True)
for path in (REPO / 'yolox/tracker').glob('*.py'):
    original = path.read_text()
    patched = re.sub(r'np\.float\b', 'float', original)
    if patched != original:
        path.write_text(patched)
demo = REPO / 'tools/demo_track.py'
source = demo.read_text()
source = source.replace('ch = cv2.waitKey(1)', 'ch = -1  # Colab has no GUI keyboard window')
source = source.replace('torch.load(ckpt_file, map_location="cpu")',
                        'torch.load(ckpt_file, map_location="cpu", weights_only=True)')
demo.write_text(source)
OUT = Path('/content/bytetrack_paper_first_run')
OUT.mkdir(exist_ok=True)
patch = subprocess.check_output(['git', '-C', str(REPO), 'diff'], text=True)
(OUT / 'compatibility.patch').write_text(patch)
print(patch)
env = dict(os.environ)
env['PYTHONPATH'] = str(REPO) + os.pathsep + env.get('PYTHONPATH', '')
subprocess.run([sys.executable, '-c',
    'from yolox.exp import get_exp; from yolox.tracker.byte_tracker import BYTETracker; print("Official imports OK")'],
    cwd=REPO, env=env, check=True)


## 4. Download the official ablation checkpoint
The file ID below comes from the official README's ablation model link. It is paired with `yolox_x_ablation.py`. The checkpoint is several hundred MB. If Google Drive rejects the automated download, use the README link to download it manually and upload it into the printed destination using Colab's Files panel, then rerun this cell.

Use only the official checkpoint. A restricted checkpoint-loading error should be inspected rather than bypassed with an unknown file. This model's documented protocol is MOT17 half-training plus CrowdHuman, followed by MOT17 half-validation. This notebook does not download or evaluate that dataset yet.


In [ ]:
import gdown
WEIGHTS = REPO / 'pretrained/bytetrack_ablation.pth.tar'
WEIGHTS.parent.mkdir(exist_ok=True)
if not WEIGHTS.exists():
    downloaded = gdown.download(id='1iqhM-6V_r1FpOlOzrdP_Ejshgk0DxOob', output=str(WEIGHTS), quiet=False)
    if downloaded is None:
        raise RuntimeError(f'Download failed. Manually upload the official checkpoint to {WEIGHTS}.')
assert WEIGHTS.is_file() and WEIGHTS.stat().st_size > 10_000_000, f'Incomplete checkpoint: {WEIGHTS}'
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
weight_hash = sha256_file(WEIGHTS)
print('Checkpoint:', WEIGHTS, 'bytes:', WEIGHTS.stat().st_size)
print('SHA256:', weight_hash, '(recorded for repeatability, not a publisher-verified checksum)')


## 5. Prepare 15 seconds of the repository's sample video
Use the supplied sample first, so you can check the installation without uploading your own footage. Later you can change SOURCE to the full path of an uploaded video. Keep the output MP4 format. This truncation is for the initial demonstration only.


In [ ]:
SOURCE = REPO / 'videos/palace.mp4'
assert SOURCE.is_file(), f'Missing input video: {SOURCE}'
INPUT = OUT / 'input_15s.mp4'
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(SOURCE), '-t', '15',
                '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(INPUT)], check=True)
import cv2
cap = cv2.VideoCapture(str(INPUT))
assert cap.isOpened(), 'Cannot decode the sample clip.'
input_info = dict(width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                  height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                  fps=float(cap.get(cv2.CAP_PROP_FPS)),
                  frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
cap.release()
print(input_info)


## 6. Run the authors' video demo
Leave settings fixed for the first run. This uses the ablation model's default image dimensions and keeps weak detections available with `--conf 0.1`. The output includes a video and MOT-style tracks. The original demo's frame numbering and no-detection behavior require review before benchmark use.


In [ ]:
command = [sys.executable, 'tools/demo_track.py', 'video',
           '-f', 'exps/example/mot/yolox_x_ablation.py',
           '-c', str(WEIGHTS), '--path', str(INPUT),
           '--device', 'gpu', '--conf', '0.1', '--track_thresh', '0.5',
           '--fp16', '--fuse', '--save_result', '-expn', 'paper_first_run']
start = time.perf_counter()
subprocess.run(command, cwd=REPO, env=env, check=True)
wall_seconds = time.perf_counter() - start
run_root = REPO / 'YOLOX_outputs/paper_first_run/track_vis'
videos = list(run_root.rglob(INPUT.name))
assert videos, 'No tracked video found. Inspect the demo output above.'
tracked_video = max(videos, key=lambda p: p.stat().st_mtime)
assert tracked_video.stat().st_size > 0
print('Saved:', tracked_video)
metadata = dict(source_revision=REVISION, checkpoint=str(WEIGHTS), checkpoint_sha256=weight_hash,
                command=command, input=input_info, input_sha256=sha256_file(INPUT),
                torch=torch.__version__, torchvision=torchvision.__version__,
                python=sys.version, gpu=torch.cuda.get_device_name(0),
                wall_seconds=wall_seconds,
                timing_scope='Entire demo subprocess including model load and output writing; not tracking-only FPS')
(OUT / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))
(OUT / 'pip_freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## 7. Watch and save the evidence
Follow a few people through the clip. Record a timestamp where an ID stays stable, changes, or disappears. These observations do not measure accuracy. Download the archive before the runtime ends.


In [ ]:
from IPython.display import Video, display
PLAYABLE = OUT / 'tracked_people.mp4'
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(tracked_video),
                '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-movflags', '+faststart',
                str(PLAYABLE)], check=True)
# Each timestamp directory has a companion TXT file in track_vis.
track_file = tracked_video.parent.parent / (tracked_video.parent.name + '.txt')
if track_file.is_file():
    shutil.copyfile(track_file, OUT / 'tracks_original_demo.txt')
else:
    print('Track text file not found at the expected location; inspect track_vis manually.')
display(Video(str(PLAYABLE), embed=True, width=900))


In [ ]:
from google.colab import files
archive = shutil.make_archive('/content/ByteTrack_official_first_run', 'zip', OUT)
print(archive)
files.download(archive)


## Your next research step

Write your own short run log: GPU, source revision, model, input, whether installation succeeded, one stable trajectory, and one failure example. Keep the compatibility patch with your experiment records. No accuracy claim should come from the sample video.

After this smoke test works, prepare MOT17 half-validation and reproduce the official evaluation protocol using the matching ablation model. Verify annotation/frame alignment and evaluation preprocessing. Then compare full ByteTrack with an explicitly controlled high-confidence-only ablation on identical detections. Report detector-plus-tracker timing separately from tracking-only timing.

Troubleshooting: if GPU selection is unavailable, try again later without buying a plan solely for this first test. If a cell fails, stop at that cell and retain its full traceback. Do not change multiple dependency versions or model files at once. The initial notebook has not been GPU-executed here, so compatibility or checkpoint-hosting issues may still arise.
